# YOLOv8 Road Damage Inference
This notebook runs image-only inference and visualizes road damage boxes on input images.

In [ ]:
import sys
import subprocess
import random
from pathlib import Path

# Install dependencies if needed
try:
    import ultralytics  # noqa: F401
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "torch", "torchvision", "pillow", "matplotlib"])

from ultralytics import YOLO
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.ops import nms

In [ ]:
# Section 1: Load Model and Dependencies
weights_path = Path("..") / "best.pt"

if not weights_path.exists():
    raise FileNotFoundError(f"Weights not found: {weights_path}. Update weights_path to your .pt file.")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLO(str(weights_path))

# Use model.predict(..., device=device) when running inference
print(f"Loaded {weights_path} on {device}")

## Load and Preprocess Input Images
Load images from disk, convert to RGB, and prepare them for inference.

In [ ]:
# Section 2: Load and Preprocess Input Images
imgsz = 640

def load_image(path: Path) -> Image.Image:
    img = Image.open(path).convert("RGB")
    return img

# Minimal preprocessing: ensure RGB and convert to numpy array
# YOLOv8 will handle resize/normalize internally, but we keep this for clarity.
def preprocess_image(img: Image.Image) -> np.ndarray:
    return np.array(img)

# # Update this to a single image you want to test
# single_image_path = Path("..") / "kaggle_output" / "runs" / "detect" / "predict" / "image1.jpg"

# if not single_image_path.exists():
#     print("Set single_image_path to a valid image file.")

# if single_image_path.exists():
#     img = load_image(single_image_path)
#     img_np = preprocess_image(img)
#     print(f"Loaded {single_image_path} with shape {img_np.shape}")

## Run Inference
Perform a forward pass and collect predictions.

In [ ]:
# Section 3: Run Inference
conf_thres = 0.25
iou_thres = 0.7

results = None
# if single_image_path.exists():
#     results = model.predict(
#         source=img_np,
#         imgsz=imgsz,
#         conf=conf_thres,
#         iou=iou_thres,
#         device=device,
#         verbose=False,
#     )
#     print("Inference complete.")

## Postprocess and Visualize Bounding Boxes
Apply thresholds and draw boxes on the image.

In [ ]:
# Section 4: Postprocess and Visualize Bounding Boxes
from PIL import ImageDraw


def draw_boxes(image: Image.Image, boxes, scores, labels, names, score_thresh=0.25):
    draw = ImageDraw.Draw(image)
    for box, score, cls_id in zip(boxes, scores, labels):
        if score < score_thresh:
            continue
        x1, y1, x2, y2 = box
        label = names.get(int(cls_id), str(int(cls_id)))
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, y1), f"{label} {score:.2f}", fill="red")
    return image

if results is not None:
    r0 = results[0]
    if r0.boxes is None or len(r0.boxes) == 0:
        print("No detections.")
    else:
        boxes_xyxy = r0.boxes.xyxy.cpu()
        scores = r0.boxes.conf.cpu()
        labels = r0.boxes.cls.cpu()

        keep = nms(boxes_xyxy, scores, iou_thres)
        boxes_xyxy = boxes_xyxy[keep].numpy()
        scores = scores[keep].numpy()
        labels = labels[keep].numpy()

        annotated = draw_boxes(img.copy(), boxes_xyxy, scores, labels, r0.names, conf_thres)
        plt.figure(figsize=(8, 6))
        plt.imshow(annotated)
        plt.axis("off")
        plt.show()

In [ ]:
# Random single-image inference from a folder
random_image_dir = Path("/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT/test/images")
random_images = list(random_image_dir.glob("*.*"))
if not random_images:
    print("No images found in random_image_dir.")
else:
    random_path = random.choice(random_images)
    img = load_image(random_path)
    img_np = preprocess_image(img)

    rand_results = model.predict(
        source=img_np,
        imgsz=imgsz,
        conf=conf_thres,
        iou=iou_thres,
        device=device,
        verbose=False,
    )
    r0 = rand_results[0]
    annotated = img.copy()
    if r0.boxes is not None and len(r0.boxes) > 0:
        boxes_xyxy = r0.boxes.xyxy.cpu()
        scores = r0.boxes.conf.cpu()
        labels = r0.boxes.cls.cpu()
        keep = nms(boxes_xyxy, scores, iou_thres)
        boxes_xyxy = boxes_xyxy[keep].numpy()
        scores = scores[keep].numpy()
        labels = labels[keep].numpy()
        annotated = draw_boxes(annotated, boxes_xyxy, scores, labels, r0.names, conf_thres)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"{random_path.name} (orig)")
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(annotated)
    plt.title(f"{random_path.name} (boxed)")
    plt.axis("off")
    plt.tight_layout()

   
    plt.savefig("randomimage.png",dpi=150)
    plt.show()
    

## Batch Inference on Multiple Images
Run inference over a folder and display a grid of results.

In [ ]:
output_path =Path("/kaggle/working/")
image_dir = Path("/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT/test/images")
image_paths = []

# for p in image_dir.glob("*.*"):
#     image_paths.append(p)
#     if len(image_paths) == 20:
#         break

for i, p in enumerate(image_dir.glob("*.*"), start=1):
    if len(image_paths) < 20:
        image_paths.append(p)
    else:
        j = random.randint(0, i - 1)
        if j < 20:
            image_paths[j] = p


if len(image_paths) == 0:
    print("Set image_dir to a folder with images.")
else:
    imgs = [load_image(p) for p in image_paths]
    imgs_np = [preprocess_image(i) for i in imgs]

    batch_results = model.predict(
        source=imgs_np,
        imgsz=imgsz,
        conf=conf_thres,
        iou=iou_thres,
        device=device,
        verbose=False,
    )

    ncols = 2
    nrows = len(image_paths)
    plt.figure(figsize=(6 * ncols, 4 * nrows))

    for i, (img_path, result) in enumerate(zip(image_paths, batch_results)):
        img_pil = load_image(img_path)
        img_pil_result = img_pil.copy()

        if result.boxes is not None and len(result.boxes) > 0:
            boxes_xyxy = result.boxes.xyxy.cpu()
            scores = result.boxes.conf.cpu()
            labels = result.boxes.cls.cpu()
            keep = nms(boxes_xyxy, scores, iou_thres)
            boxes_xyxy = boxes_xyxy[keep].numpy()
            scores = scores[keep].numpy()
            labels = labels[keep].numpy()
            img_pil_result = draw_boxes(img_pil_result, boxes_xyxy, scores, labels, result.names, conf_thres)

        plt.subplot(nrows, ncols, (2 * i) + 1)
        plt.imshow(img_pil)
        plt.title(f"{img_path.name} (orig)")
        plt.axis("off")

        plt.subplot(nrows, ncols, (2 * i) + 2)
        plt.imshow(img_pil_result)
        plt.title(f"{img_path.name} (boxed)")
        plt.axis("off")

    plt.tight_layout()
    plt.savefig("inference_test.png")
    plt.show()
    